In [ ]:
#| default_exp flakes

In [ ]:
# greet('dutil')

**NOTE**: the import block below should be tagged for export, but is left unexported for demonstration purposes.

In [ ]:
#| export
import re
from collections import defaultdict
from io import StringIO
from pathlib import Path
from IPython.display import Markdown
from itertools import repeat
from urllib.parse import quote_plus
import fastcore.all as FC
import pyflakes
from pyflakes.reporter import Reporter
from pyflakes.messages import Message
from pyflakes.api import check as _check
from dialoghelper.core import find_msgs, find_dname
from pote.dutil import link_msg
from pote.dialog import is_exported

In [ ]:
from IPython.display import Markdown
from itertools import accumulate, repeat

import nbdev.config
import fastcore.all as FC
from rich.console import Console
from fastcore.test import *
from fasthtml.components import Button
from dialoghelper.stdtools import *
from dialoghelper.core import is_usable_tool

In [ ]:
cprint = Console(width=140, tab_size=4, force_jupyter=True if FC.IN_IPYTHON else None).print

## setup - install pyflakes

**NOTE**: not cool, no one should be obligued to install a package just for perusing a notebook. There's nothing I can do for now -> see exploration [uv run & PEP 876](/dialog_?name=vic/explorer/uv)

`pyflakes` is a very lightweight, zero dependencies, python package.

In [ ]:
!pip install -U pyflakes

### download companion dialog

To follow this dialog, load the [test dialog](https://raw.githubusercontent.com/civvic/dutil/refs/heads/main/nbs/data/test.ipynb) and place it in disk next to this dialog. Edit the following variables to point to that, or one of your dialogs.

In [ ]:
Path.home()/(find_dname('data/test').removeprefix('/'))

Path('/app/data/prj/pote/nbs/data/test')

In [ ]:
_DREL = 'data/test'
_DABS = find_dname(_DREL)
assert (Path.home()/_DABS[1:]).with_suffix('.ipynb').exists()
_DREL, _DABS

('data/test', '/prj/pote/nbs/data/test')

# flakes

I'm working on **dutil**, a nbdev project. I want to create a utility function that checks for import issues in exported code cells before running `nbdev_export()`.

The problem: When developing in a notebook/dialog, code can work fine because symbols are available in the global scope, but when exported to a module, missing imports cause failures. This is especially problematic inside function bodies where a symbol might not be imported.

Solution approach: Use `pyflakes` to check the concatenated exported code cells and report undefined names. This gives us a lightweight way to catch import (and many other) issues without running a full lint/type checker.

We'll need to:
1. Extract all exported code cells from the current dialog
2. Concatenate them into a string
3. Run pyflakes on it
4. Return structured warnings

## API usage

In [ ]:
cds = await find_msgs(msg_type='code')
len(cds)

89

In [ ]:
src = '\n'.join(cds.filter(lambda x: x.is_exported).itemgot('content'))
print(src[:500])

class ListReporter(Reporter):
    def __init__(self):
        self.warnings = FC.L(); super().__init__(None, None)
        super().__init__(StringIO(), StringIO())
    def flake(self, message): self.warnings.append(message.as_tuple())
    def close(self): self._stderr.close(); self._stdout.close()
    def unexpectedError(self, filename, msg): self.warnings.append(('UnexpectedError', filename, (0,0), msg, ()))
    def syntaxError(self, filename, msg, lineno, offset, text):
        line = None if 


In [ ]:
%%capture captured_output
srcname = '05_flakes.py'
_check(src, srcname)

In [ ]:
print(captured_output.stdout[:200])

05_flakes.py:1:20: undefined name 'Reporter'
05_flakes.py:3:25: undefined name 'FC'
05_flakes.py:4:26: undefined name 'StringIO'
05_flakes.py:4:38: undefined name 'StringIO'
05_flakes.py:14:27: undefi


In [ ]:

fn_rng, warn = "05_flakes.py:2:2: undefined name 'FC'".split(': ')
fn, _, rng = fn_rng.partition(':')
fn, rng, warn

('05_flakes.py', '2:2', "undefined name 'FC'")

## custom reporter

In [ ]:
def UnexpectedError(Message):
    def __init__(self, filename, loc, name): Message.__init__(self, filename, loc)


In [ ]:
#| exporti
@FC.patch
def as_tuple(self: Message): return type(self).__name__, self.filename, (self.lineno, self.col+1), self.message % self.message_args, self.message_args

In [ ]:
#| export
class ListReporter(Reporter):
    def __init__(self):
        self.warnings = FC.L(); super().__init__(None, None)
        super().__init__(StringIO(), StringIO())
    def flake(self, message): self.warnings.append(message.as_tuple())
    def close(self): self._stderr.close(); self._stdout.close()
    def unexpectedError(self, filename, msg): self.warnings.append(('UnexpectedError', filename, (0,0), msg, ()))
    def syntaxError(self, filename, msg, lineno, offset, text):
        line = None if text is None else text.splitlines()[-1]
        lineno = max(lineno or 0, 1)
        if offset is not None: offset = max(offset, 1)
        rng, s = (lineno, offset or 0), ''
        if line is not None:
           s = f"{line}\n{re.sub(r'\S', ' ', line[:offset-1])+"^\n" if offset is not None else ''}"
        self.warnings.append(('SyntaxError', filename, rng, msg, (s,)))

In [ ]:
_check(src, srcname, reprt := ListReporter())

21

In [ ]:
for _ in reprt.warnings[:5]: print(_)

('UndefinedName', '05_flakes.py', (1, 20), "undefined name 'Reporter'", ('Reporter',))
('UndefinedName', '05_flakes.py', (3, 25), "undefined name 'FC'", ('FC',))
('UndefinedName', '05_flakes.py', (4, 26), "undefined name 'StringIO'", ('StringIO',))
('UndefinedName', '05_flakes.py', (4, 38), "undefined name 'StringIO'", ('StringIO',))
('UndefinedName', '05_flakes.py', (14, 27), "undefined name 're'", ('re',))


In [ ]:
src = '''from dialoghelper.core import read_msg, find_msg_id
from dialoghelper.inspecttools import resolve
def get_output(msgid:str=None) -> list[str]:
    await read_msg(n=0, relative=True, msgid=msgid or find_msg_id())
    if o := resolve('msg.output'):
        if isinstance(o, str): return o  # prompt
    if isinstance(o, str): return [o]
'''
_check(src, '_', reprt := ListReporter())
for _ in reprt.warnings: print(_)

In [ ]:
src = '''1a = 10'''
_check(src, '_', reprt := ListReporter())
for _ in reprt.warnings: print(_)

('SyntaxError', '_', (1, 1), 'invalid decimal literal', ('1a = 10\n^\n',))


In [ ]:
src = '''check_source_flakes??'''
_check(src, '_', reprt := ListReporter())
for _ in reprt.warnings: print(_)

('SyntaxError', '_', (1, 20), 'invalid syntax', ('check_source_flakes??\n                   ^\n',))


In [ ]:
src = '''!ls -la'''
_check(src, '_', reprt := ListReporter())
for _ in reprt.warnings: print(_)

('SyntaxError', '_', (1, 1), 'invalid syntax', ('!ls -la\n^\n',))


## v1

In [ ]:
async def check_flakes(src:str=None, srcname:str=None):
    if not src:
        cds = await find_msgs(msg_type='code')
        src = '\n'.join(cds.filter(lambda x: x.is_exported).itemgot('content'))
    if not srcname:
        srcname = (Path.home()/find_dname()).with_suffix('.py').name
    _check(src, srcname, reprt := ListReporter())
    return reprt.warnings
    

In [ ]:
for wrn in (await check_flakes(src, srcname))[:5]: print(wrn)

('SyntaxError', '05_flakes.py', (1, 1), 'invalid syntax', ('!ls -la\n^\n',))


In [ ]:
for wrn in (await check_flakes())[:5]: print(wrn)

('UndefinedName', '01_flakes.py', (1, 20), "undefined name 'Reporter'", ('Reporter',))
('UndefinedName', '01_flakes.py', (3, 25), "undefined name 'FC'", ('FC',))
('UndefinedName', '01_flakes.py', (4, 26), "undefined name 'StringIO'", ('StringIO',))
('UndefinedName', '01_flakes.py', (4, 38), "undefined name 'StringIO'", ('StringIO',))
('UndefinedName', '01_flakes.py', (14, 27), "undefined name 're'", ('re',))


## v2

In [ ]:
nbdev.config.is_nbdev(), nbdev.config.get_config().lib_path


(True, Path('/app/data/prj/pote/pote'))

In [ ]:
('#| hide'.strip()[2:]).strip().split()

['hide']

In [ ]:
async def check_flakes(
    src:str=None,
    srcname:str=None, 
):
    "Check for pyflakes 'warnings' in `src` or current dialog"
    if not src:
        cds = await find_msgs(msg_type='code')
        src = '\n'.join(cds.filter(lambda x: x.is_exported).itemgot('content'))
        if not srcname and nbdev.config.is_nbdev():
            if s := FC.first(m.content for m in cds if m.content and re.match(r'\s*#\s*\|', m.content.split()[0])):
                dirct = (s.strip()[2:]).strip().split()
                if dirct[0] == 'default_exp' and len(dirct) > 1: srcname = f"{dirct[1]}.py"
    if not srcname:
        srcname = (Path.home()/find_dname()).with_suffix('.py').name
    _check(src, srcname, reprt := ListReporter())
    return reprt.warnings
    

In [ ]:
for wrn in (await check_flakes())[:5]: print(wrn)

('UndefinedName', 'flakes.py', (1, 20), "undefined name 'Reporter'", ('Reporter',))
('UndefinedName', 'flakes.py', (3, 25), "undefined name 'FC'", ('FC',))
('UndefinedName', 'flakes.py', (4, 26), "undefined name 'StringIO'", ('StringIO',))
('UndefinedName', 'flakes.py', (4, 38), "undefined name 'StringIO'", ('StringIO',))
('UndefinedName', 'flakes.py', (14, 27), "undefined name 're'", ('re',))


## line 2 msgid

In [ ]:
def _code_with_msgid(codes, msgids):
    "Concatenate code blocks with msgids and track line ranges"
    s, line, l2id = '', 1, {}
    for code, msgid in zip(codes, msgids):
        s += f"{'\n' if line > 1 else ''}# %% msgid: {msgid}\n" + code
        end = line + code.count('\n') + 1
        l2id |= dict(product(range(line, end), (msgid,)))
        line = end
    l2id |= dict(product(range(line, len(s.splitlines())+1), (msgid,)))
    return s, l2id


In [ ]:
codes = ['import os\n\nprint("hi")', 'def foo():\n    return 42', 'x = foo()\nprint(x)']
msgids = ['_abc123', '_def456', '_ghi789']

code, l2id = _code_with_msgid(codes, msgids)
cprint('\n'.join(f"{i+1:3} {line}" for i,line in enumerate(code.split('\n'))))
cprint('\n',l2id)

1 # %% msgid: _abc123
  2 import os
  3 
  4 print("hi")
  5 # %% msgid: _def456
  6 def foo():
  7     return 42
  8 # %% msgid: _ghi789
  9 x = foo()
 10 print(x)

{
    1: '_abc123',
    2: '_abc123',
    3: '_abc123',
    4: '_def456',
    5: '_def456',
    6: '_ghi789',
    7: '_ghi789',
    8: '_ghi789',
    9: '_ghi789',
    10: '_ghi789'
}

In [ ]:
try:
    test_eq(l2id[3], '_abc123')
    test_eq(l2id[5], '_abc123')
    test_eq(l2id[6], '_def456')
    test_eq(l2id[10], '_ghi789')
except Exception as e:
    display(e)

AssertionError('==:\n_def456\n_abc123')

In [ ]:
s = '\n\n'.join(f"# %% msgid: {msgid}\n{code}" for code, msgid in zip(codes, msgids))
cprint('\n'.join(f"{i+1:3} {line}" for i,line in enumerate(s.split('\n'))))

1 # %% msgid: _abc123
  2 import os
  3 
  4 print("hi")
  5 
  6 # %% msgid: _def456
  7 def foo():
  8     return 42
  9 
 10 # %% msgid: _ghi789
 11 x = foo()
 12 print(x)

In [ ]:
line = 1
code, msgid = codes[0], msgids[0]
end = line + code.count('\n') + 3  # base 1, + 2 cr
end

6

In [ ]:
line = 6
code, msgid = codes[1], msgids[1]
end = line + code.count('\n') + 3  # base 1, + 2 cr
end

10

In [ ]:
line = 10
code, msgid = codes[2], msgids[2]
end = min(s.count('\n')+1, line + code.count('\n') + 3)  # base 1, + 2 cr
end

12

In [ ]:
line, ls, l2id = 1, s.count('\n')+1, {}
for code, msgid in zip(codes, msgids):
    end = min(ls+1, line + code.count('\n') + 3)
    l2id |= {l:m for l,m in product(range(line, end), ((msgid, line),))}
    line = end
print(l2id)


{1: ('_abc123', 1), 2: ('_abc123', 1), 3: ('_abc123', 1), 4: ('_abc123', 1), 5: ('_abc123', 1), 6: ('_def456', 6), 7: ('_def456', 6), 8: ('_def456', 6), 9: ('_def456', 6), 10: ('_ghi789', 10), 11: ('_ghi789', 10), 12: ('_ghi789', 10)}


In [ ]:
cprint('\n'.join(f"{i+1:3} {line}" for i,line in enumerate(s.split('\n'))))

1 # %% msgid: _abc123
  2 import os
  3 
  4 print("hi")
  5 
  6 # %% msgid: _def456
  7 def foo():
  8     return 42
  9 
 10 # %% msgid: _ghi789
 11 x = foo()
 12 print(x)

In [ ]:
line, ls, l2id = 1, s.count('\n')+1, {}
for code, msgid in zip(codes, msgids):
    end = min(ls+1, line + code.count('\n') + 3)
    l2id |= {l:m for l,m in zip(range(line, end), product(range(1, len(code)+2), (msgid,)))}
    line = end
print(l2id)


{1: (1, '_abc123'), 2: (2, '_abc123'), 3: (3, '_abc123'), 4: (4, '_abc123'), 5: (5, '_abc123'), 6: (1, '_def456'), 7: (2, '_def456'), 8: (3, '_def456'), 9: (4, '_def456'), 10: (1, '_ghi789'), 11: (2, '_ghi789'), 12: (3, '_ghi789')}


In [ ]:
#| export
def _line2msgid(codes, msgids):
    "Return list of (code_line, line_num_in_msg, msgid) tuples for each concatenated source line"
    line, l2id = 1, FC.L()
    for code, msgid in zip(codes, msgids):
        codelns = f"# %% msgid: {msgid}\n{code}\n".split('\n')
        l2id.extend(zip(codelns, range(0, len(codelns)-1), repeat(msgid)))
        line += len(codelns)
    return l2id

In [ ]:
l2id = _line2msgid(codes, msgids)
print('\n'.join(f"{i+1:2}: {l[1]} {l[2]} - {l[0]}" for i, l in l2id.enumerate()))
src = '\n'.join(l2id.itemgot(0))
print(src)

 1: 0 _abc123 - # %% msgid: _abc123
 2: 1 _abc123 - import os
 3: 2 _abc123 - 
 4: 3 _abc123 - print("hi")
 5: 0 _def456 - # %% msgid: _def456
 6: 1 _def456 - def foo():
 7: 2 _def456 -     return 42
 8: 0 _ghi789 - # %% msgid: _ghi789
 9: 1 _ghi789 - x = foo()
10: 2 _ghi789 - print(x)
# %% msgid: _abc123
import os

print("hi")
# %% msgid: _def456
def foo():
    return 42
# %% msgid: _ghi789
x = foo()
print(x)


`_line2msgid` returns a list of tuples used downstream as a simple dataa structure to handle efficiently pyflakes warnings. 

1. `_line2msgid` returns a list of tuples: `(code_line, line_num_in_msg, msgid)` for each line
2. Source given to pyflakes comes from this: `'\n'.join(l2id.itemgot(0))`
3. When pyflakes returns a warning with a line number (say line 6), we look up `l2id[5]` (0-indexed) to get `(code_line, line_num_in_msg, msgid)` to get both the msgid and the exact line within that message where the issue occurred.

## code messages

In [ ]:
Path(find_dname()).name


'01_flakes'

In [ ]:
(Path.home()/find_dname()).with_suffix('.ipynb')

Path('/prj/pote/nbs/01_flakes.ipynb')

In [ ]:
dname = _DABS

dname

'/prj/pote/nbs/data/test'

In [ ]:
cds = await find_msgs(msg_type='code', dname=dname)
exptd = cds.filter(lambda x: x.is_exported)
for _ in exptd.attrgot('content'): cprint(_[:100])

import re
import sys
import inspect
from typing import Any
import fastcore.all as FC
from fastcore.x

def solveit_version():
    "Return the version of solveit if it is found"
    s = ' '.join(_.__modul

def in_dialog():
    "Check if the code is running in a solveit dialog"
    return bool(solveit_vers

def get_caller_globals(): 
    "Return the globals of the caller"
    return inspect.currentframe().

_empty = inspect.Parameter.empty

def at_(
    o, # Object to traverse (dict, list, object, or nested combination)
    sym: str, # Pat

def get_tool_names(ns=None, exclude=None, only_exported=False, exclude_private=True):
    "Return th

def add_tools_card(ns=None):
    "Add a message with all tools in namespace `ns` or caller globals"

def setup_ns(ns=None, **kwargs):
    "Add `kwargs` to the namespace `ns` or current dialog"
    ns =

In [ ]:
len(cds), len(exptd)

(80, 9)

In [ ]:
#| export
async def get_tagged_source(
    msgs:list=None,  # list of message dicts with 'content' and 'id' keys
    all:bool=False,  # include all messages, not just exported ones
    dname:str=''  # dialog name (relative like 'data/test' or absolute, starting with '/'), defaults to current
) -> tuple[str, list]:  # Returns (source, line_mapping) where line_mapping maps each source line to (code_line, line_in_msg, msgid)
    "Concatenate message contents with msgid markers"
    if not msgs:
        msgs = await find_msgs(msg_type='code', dname=dname)
        msgs = msgs if all else msgs.filter(is_exported)
    l2id = _line2msgid(msgs.attrgot('content'), msgs.attrgot('id'))
    return '\n'.join(l2id.itemgot(0)), l2id

In [ ]:
src, l2id = await get_tagged_source(exptd)

cprint(src[:400])

# %% msgid: _e72f67fd
import re
import sys
import inspect
from typing import Any
import fastcore.all as FC
from fastcore.xtras import is_listy
import dialoghelper
from dialoghelper import *
from dialoghelper.core import _find_frame_dict
# %% msgid: _6389d58d
def solveit_version():
    "Return the version of solveit if it is found"
    s = ' '.join(_.__module__ for _ in sys.meta_path)
    mtch = re

In [ ]:
src, l2id = await get_tagged_source()
print('----', src[:400])

---- # %% msgid: _42c1d335
#| exporti
@FC.patch
def as_tuple(self: Message): return type(self).__name__, self.filename, (self.lineno, self.col+1), self.message % self.message_args, self.message_args
# %% msgid: _b6e1684c
class ListReporter(Reporter):
    def __init__(self):
        self.warnings = FC.L(); super().__init__(None, None)
        super().__init__(StringIO(), StringIO())
    def flake(self, 


## warning types

In [ ]:
pyflakes.messages??


```python
"""
Provide the class Message and its subclasses.
"""


class Message:
    message = ''
    message_args = ()

    def __init__(self, filename, loc):
        self.filename = filename
        self.lineno = loc.lineno
        self.col = loc.col_offset

    def __str__(self):
        return '{}:{}:{}: {}'.format(self.filename, self.lineno, self.col+1,
                                     self.message % self.message_args)


class UnusedImport(Message):
    message = '%r imported but unused'

    def __init__(self, filename, loc, name):
        Message.__init__(self, filename, loc)
        self.message_args = (name,)


class RedefinedWhileUnused(Message):
    message = 'redefinition of unused %r from line %r'

    def __init__(self, filename, loc, name, orig_loc):
        Message.__init__(self, filename, loc)
        self.message_args = (name, orig_loc.lineno)


class ImportShadowedByLoopVar(Message):
    message = 'import %r from line %r shadowed by loop variable'

    def __init__(self, filename, loc, name, orig_loc):
        Message.__init__(self, filename, loc)
        self.message_args = (name, orig_loc.lineno)


class ImportStarNotPermitted(Message):
    message = "'from %s import *' only allowed at module level"

    def __init__(self, filename, loc, modname):
        Message.__init__(self, filename, loc)
        self.message_args = (modname,)


class ImportStarUsed(Message):
    message = "'from %s import *' used; unable to detect undefined names"

    def __init__(self, filename, loc, modname):
        Message.__init__(self, filename, loc)
        self.message_args = (modname,)


class ImportStarUsage(Message):
    message = "%r may be undefined, or defined from star imports: %s"

    def __init__(self, filename, loc, name, from_list):
        Message.__init__(self, filename, loc)
        self.message_args = (name, from_list)


class UndefinedName(Message):
    message = 'undefined name %r'

    def __init__(self, filename, loc, name):
        Message.__init__(self, filename, loc)
        self.message_args = (name,)


class DoctestSyntaxError(Message):
    message = 'syntax error in doctest'

    def __init__(self, filename, loc, position=None):
        Message.__init__(self, filename, loc)
        if position:
            (self.lineno, self.col) = position
        self.message_args = ()


class UndefinedExport(Message):
    message = 'undefined name %r in __all__'

    def __init__(self, filename, loc, name):
        Message.__init__(self, filename, loc)
        self.message_args = (name,)


class UndefinedLocal(Message):
    message = 'local variable %r {0} referenced before assignment'

    default = 'defined in enclosing scope on line %r'
    builtin = 'defined as a builtin'

    def __init__(self, filename, loc, name, orig_loc):
        Message.__init__(self, filename, loc)
        if orig_loc is None:
            self.message = self.message.format(self.builtin)
            self.message_args = name
        else:
            self.message = self.message.format(self.default)
            self.message_args = (name, orig_loc.lineno)


class DuplicateArgument(Message):
    message = 'duplicate argument %r in function definition'

    def __init__(self, filename, loc, name):
        Message.__init__(self, filename, loc)
        self.message_args = (name,)


class MultiValueRepeatedKeyLiteral(Message):
    message = 'dictionary key %r repeated with different values'

    def __init__(self, filename, loc, key):
        Message.__init__(self, filename, loc)
        self.message_args = (key,)


class MultiValueRepeatedKeyVariable(Message):
    message = 'dictionary key variable %s repeated with different values'

    def __init__(self, filename, loc, key):
        Message.__init__(self, filename, loc)
        self.message_args = (key,)


class LateFutureImport(Message):
    message = 'from __future__ imports must occur at the beginning of the file'


class FutureFeatureNotDefined(Message):
    """An undefined __future__ feature name was imported."""
    message = 'future feature %s is not defined'

    def __init__(self, filename, loc, name):
        Message.__init__(self, filename, loc)
        self.message_args = (name,)


class UnusedVariable(Message):
    """
    Indicates that a variable has been explicitly assigned to but not actually
    used.
    """
    message = 'local variable %r is assigned to but never used'

    def __init__(self, filename, loc, names):
        Message.__init__(self, filename, loc)
        self.message_args = (names,)


class UnusedAnnotation(Message):
    """
    Indicates that a variable has been explicitly annotated to but not actually
    used.
    """
    message = 'local variable %r is annotated but never used'

    def __init__(self, filename, loc, names):
        Message.__init__(self, filename, loc)
        self.message_args = (names,)


class UnusedIndirectAssignment(Message):
    """A `global` or `nonlocal` statement where the name is never reassigned"""
    message = '`%s %s` is unused: name is never assigned in scope'

    def __init__(self, filename, loc, name):
        Message.__init__(self, filename, loc)
        self.message_args = (type(loc).__name__.lower(), name)


class ReturnOutsideFunction(Message):
    """
    Indicates a return statement outside of a function/method.
    """
    message = '\'return\' outside function'


class YieldOutsideFunction(Message):
    """
    Indicates a yield or yield from statement outside of a function/method.
    """
    message = '\'yield\' outside function'


# For whatever reason, Python gives different error messages for these two. We
# match the Python error message exactly.
class ContinueOutsideLoop(Message):
    """
    Indicates a continue statement outside of a while or for loop.
    """
    message = '\'continue\' not properly in loop'


class BreakOutsideLoop(Message):
    """
    Indicates a break statement outside of a while or for loop.
    """
    message = '\'break\' outside loop'


class DefaultExceptNotLast(Message):
    """
    Indicates an except: block as not the last exception handler.
    """
    message = 'default \'except:\' must be last'


class TwoStarredExpressions(Message):
    """
    Two or more starred expressions in an assignment (a, *b, *c = d).
    """
    message = 'two starred expressions in assignment'


class TooManyExpressionsInStarredAssignment(Message):
    """
    Too many expressions in an assignment with star-unpacking
    """
    message = 'too many expressions in star-unpacking assignment'


class IfTuple(Message):
    """
    Conditional test is a non-empty tuple literal, which are always True.
    """
    message = '\'if tuple literal\' is always true, perhaps remove accidental comma?'


class AssertTuple(Message):
    """
    Assertion test is a non-empty tuple literal, which are always True.
    """
    message = 'assertion is always true, perhaps remove parentheses?'


class ForwardAnnotationSyntaxError(Message):
    message = 'syntax error in forward annotation %r'

    def __init__(self, filename, loc, annotation):
        Message.__init__(self, filename, loc)
        self.message_args = (annotation,)


class RaiseNotImplemented(Message):
    message = "'raise NotImplemented' should be 'raise NotImplementedError'"


class InvalidPrintSyntax(Message):
    message = 'use of >> is invalid with print function'


class IsLiteral(Message):
    message = 'use ==/!= to compare constant literals (str, bytes, int, float, tuple)'


class FStringMissingPlaceholders(Message):
    message = 'f-string is missing placeholders'


class TStringMissingPlaceholders(Message):
    message = 't-string is missing placeholders'


class StringDotFormatExtraPositionalArguments(Message):
    message = "'...'.format(...) has unused arguments at position(s): %s"

    def __init__(self, filename, loc, extra_positions):
        Message.__init__(self, filename, loc)
        self.message_args = (extra_positions,)


class StringDotFormatExtraNamedArguments(Message):
    message = "'...'.format(...) has unused named argument(s): %s"

    def __init__(self, filename, loc, extra_keywords):
        Message.__init__(self, filename, loc)
        self.message_args = (extra_keywords,)


class StringDotFormatMissingArgument(Message):
    message = "'...'.format(...) is missing argument(s) for placeholder(s): %s"

    def __init__(self, filename, loc, missing_arguments):
        Message.__init__(self, filename, loc)
        self.message_args = (missing_arguments,)


class StringDotFormatMixingAutomatic(Message):
    message = "'...'.format(...) mixes automatic and manual numbering"


class StringDotFormatInvalidFormat(Message):
    message = "'...'.format(...) has invalid format string: %s"

    def __init__(self, filename, loc, error):
        Message.__init__(self, filename, loc)
        self.message_args = (error,)


class PercentFormatInvalidFormat(Message):
    message = "'...' %% ... has invalid format string: %s"

    def __init__(self, filename, loc, error):
        Message.__init__(self, filename, loc)
        self.message_args = (error,)


class PercentFormatMixedPositionalAndNamed(Message):
    message = "'...' %% ... has mixed positional and named placeholders"


class PercentFormatUnsupportedFormatCharacter(Message):
    message = "'...' %% ... has unsupported format character %r"

    def __init__(self, filename, loc, c):
        Message.__init__(self, filename, loc)
        self.message_args = (c,)


class PercentFormatPositionalCountMismatch(Message):
    message = "'...' %% ... has %d placeholder(s) but %d substitution(s)"

    def __init__(self, filename, loc, n_placeholders, n_substitutions):
        Message.__init__(self, filename, loc)
        self.message_args = (n_placeholders, n_substitutions)


class PercentFormatExtraNamedArguments(Message):
    message = "'...' %% ... has unused named argument(s): %s"

    def __init__(self, filename, loc, extra_keywords):
        Message.__init__(self, filename, loc)
        self.message_args = (extra_keywords,)


class PercentFormatMissingArgument(Message):
    message = "'...' %% ... is missing argument(s) for placeholder(s): %s"

    def __init__(self, filename, loc, missing_arguments):
        Message.__init__(self, filename, loc)
        self.message_args = (missing_arguments,)


class PercentFormatExpectedMapping(Message):
    message = "'...' %% ... expected mapping but got sequence"


class PercentFormatExpectedSequence(Message):
    message = "'...' %% ... expected sequence but got mapping"


class PercentFormatStarRequiresSequence(Message):
    message = "'...' %% ... `*` specifier requires sequence"
```

**File:** `~/.local/lib/python3.12/site-packages/pyflakes/messages.py`

In [ ]:
pfmod = pyflakes.messages
syms = [_ for _ in dir(pfmod) if _[0] != '_']
    

In [ ]:
kls_msg = pfmod.Message
kls = getattr(pfmod, 'UnusedVariable')
kls_msg in kls.mro()

True

In [ ]:
[sym for sym in syms if isinstance(o := getattr(pfmod, sym), type) and kls_msg in o.mro()]
        

['AssertTuple',
 'BreakOutsideLoop',
 'ContinueOutsideLoop',
 'DefaultExceptNotLast',
 'DoctestSyntaxError',
 'DuplicateArgument',
 'FStringMissingPlaceholders',
 'ForwardAnnotationSyntaxError',
 'FutureFeatureNotDefined',
 'IfTuple',
 'ImportShadowedByLoopVar',
 'ImportStarNotPermitted',
 'ImportStarUsage',
 'ImportStarUsed',
 'InvalidPrintSyntax',
 'IsLiteral',
 'LateFutureImport',
 'Message',
 'MultiValueRepeatedKeyLiteral',
 'MultiValueRepeatedKeyVariable',
 'PercentFormatExpectedMapping',
 'PercentFormatExpectedSequence',
 'PercentFormatExtraNamedArguments',
 'PercentFormatInvalidFormat',
 'PercentFormatMissingArgument',
 'PercentFormatMixedPositionalAndNamed',
 'PercentFormatPositionalCountMismatch',
 'PercentFormatStarRequiresSequence',
 'PercentFormatUnsupportedFormatCharacter',
 'RaiseNotImplemented',
 'RedefinedWhileUnused',
 'ReturnOutsideFunction',
 'StringDotFormatExtraNamedArguments',
 'StringDotFormatExtraPositionalArguments',
 'StringDotFormatInvalidFormat',
 'StringDot

In [ ]:
pfmod.UnusedVariable.__doc__

'\n    Indicates that a variable has been explicitly assigned to but not actually\n    used.\n    '

In [ ]:
#| export
def _warningtypes():
    "Return all pyflakes message types"
    pfmod = pyflakes.messages
    kls_msg = pfmod.Message
    syms = [_ for _ in dir(pfmod) if _[0] != '_']
    return {sym: o.__doc__.strip() if o.__doc__ else o.message
        for sym in syms 
        if sym != 'Message' and isinstance(o := getattr(pfmod, sym), type) and kls_msg in o.mro()}

In [ ]:
#| export
_ALL = ','.join(_warningtypes().keys())
IMPORTS = 'ImportShadowedByLoopVar,ImportStarNotPermitted,ImportStarUsage,ImportStarUsed,LateFutureImport,UnusedImport'
VARS = 'UndefinedExport,UndefinedLocal,UndefinedName,UnusedIndirectAssignment,UnusedVariable'


In [ ]:
_ALL

'AssertTuple,BreakOutsideLoop,ContinueOutsideLoop,DefaultExceptNotLast,DoctestSyntaxError,DuplicateArgument,FStringMissingPlaceholders,ForwardAnnotationSyntaxError,FutureFeatureNotDefined,IfTuple,ImportShadowedByLoopVar,ImportStarNotPermitted,ImportStarUsage,ImportStarUsed,InvalidPrintSyntax,IsLiteral,LateFutureImport,MultiValueRepeatedKeyLiteral,MultiValueRepeatedKeyVariable,PercentFormatExpectedMapping,PercentFormatExpectedSequence,PercentFormatExtraNamedArguments,PercentFormatInvalidFormat,PercentFormatMissingArgument,PercentFormatMixedPositionalAndNamed,PercentFormatPositionalCountMismatch,PercentFormatStarRequiresSequence,PercentFormatUnsupportedFormatCharacter,RaiseNotImplemented,RedefinedWhileUnused,ReturnOutsideFunction,StringDotFormatExtraNamedArguments,StringDotFormatExtraPositionalArguments,StringDotFormatInvalidFormat,StringDotFormatMissingArgument,StringDotFormatMixingAutomatic,TStringMissingPlaceholders,TooManyExpressionsInStarredAssignment,TwoStarredExpressions,Undefined

## v3

In [ ]:
#| export
def check_source_flakes(
    src:str,  # source code, typically concatenated exported code cells
    src_name:str='',  # name of source (needed by pyflakes)
) -> list:  # Returns list of (type, name, (line,col), msg, args) tuples
    "Check for pyflakes 'warnings' in `src`"
    _check(src, src_name or 'synthetic', reprt := ListReporter())
    return reprt.warnings

In [ ]:
src, _ = await get_tagged_source(exptd)
check_source_flakes(src)

[('UnusedImport', 'synthetic', (6, 1), "'fastcore.all as FC' imported but unused", ('fastcore.all as FC',)), ('ImportStarUsed', 'synthetic', (9, 1), "'from dialoghelper import *' used; unable to detect undefined names", ('dialoghelper',)), ('ImportStarUsage', 'synthetic', (62, 20), "'is_usable_tool' may be undefined, or defined from star imports: dialoghelper", ('is_usable_tool', 'dialoghelper')), ('ImportStarUsage', 'synthetic', (68, 5), "'add_msg' may be undefined, or defined from star imports: dialoghelper", ('add_msg', 'dialoghelper')), ('ImportStarUsage', 'synthetic', (68, 13), "'mk_toollist' may be undefined, or defined from star imports: dialoghelper", ('mk_toollist', 'dialoghelper')), ('ImportStarUsage', 'synthetic', (74, 14), "'find_msg_id' may be undefined, or defined from star imports: dialoghelper", ('find_msg_id', 'dialoghelper')), ('ImportStarUsage', 'synthetic', (76, 13), "'add_msg' may be undefined, or defined from star imports: dialoghelper", ('add_msg', 'dialoghelper'

In [ ]:
is_usable_tool(check_source_flakes)

True

In [ ]:
async def check_flakes(
    dname:str='',  # dialog name (relative like 'data/test' or absolute), defaults to current dialog
) -> list:  # Returns list of dicts with keys: wtype (warning category), description, id (code message id where issue occurs), line_n (line number in message, 1-indexed)
    "Check exported code for import/variable/general issues, tipically before nbdev_export. Example: check_flakes() returns [{wtype:'UnusedImport', description:'os imported but unused', id:'_abc123', line_n:5}]"
    cds = await find_msgs(msg_type='code', dname=dname)
    exptd = cds.filter(lambda x: x.is_exported)
    src, l2id = await get_tagged_source(exptd)
    ws = check_source_flakes(src, Path(dname or find_dname()).name)
    return [dict(wtype=w[0], description=w[3], id=l2id[w[2][0]-1][2], line_n=l2id[w[2][0]-1][1]) for w in ws]

In [ ]:
wrns = await check_flakes()
for _ in wrns[:5]: print(_)

{'wtype': 'UndefinedName', 'description': "undefined name 'Reporter'", 'id': '_b6e1684c', 'line_n': 1}
{'wtype': 'UndefinedName', 'description': "undefined name 'FC'", 'id': '_b6e1684c', 'line_n': 3}
{'wtype': 'UndefinedName', 'description': "undefined name 'StringIO'", 'id': '_b6e1684c', 'line_n': 4}
{'wtype': 'UndefinedName', 'description': "undefined name 'StringIO'", 'id': '_b6e1684c', 'line_n': 4}
{'wtype': 'UndefinedName', 'description': "undefined name 're'", 'id': '_b6e1684c', 'line_n': 14}


In [ ]:
wrns = await check_flakes(dname=_DREL)
for _ in wrns[:5]: print(_)

{'wtype': 'UnusedImport', 'description': "'fastcore.all as FC' imported but unused", 'id': '_e72f67fd', 'line_n': 5}
{'wtype': 'ImportStarUsed', 'description': "'from dialoghelper import *' used; unable to detect undefined names", 'id': '_e72f67fd', 'line_n': 8}
{'wtype': 'ImportStarUsage', 'description': "'is_usable_tool' may be undefined, or defined from star imports: dialoghelper", 'id': '_9027bb28', 'line_n': 13}
{'wtype': 'ImportStarUsage', 'description': "'add_msg' may be undefined, or defined from star imports: dialoghelper", 'id': '_612c72ae', 'line_n': 4}
{'wtype': 'ImportStarUsage', 'description': "'mk_toollist' may be undefined, or defined from star imports: dialoghelper", 'id': '_612c72ae', 'line_n': 4}


In [ ]:
is_usable_tool(check_flakes)

True

## filter

In [ ]:
#| export
async def _get_flakes(dname='', wtypes='', all=False):
    src, l2id = await get_tagged_source(all=all, dname=dname)
    ws = check_source_flakes(src, Path(dname or find_dname()).name)
    if wtypes: 
        ts = set(FC.L(wtypes.split(',')).map(str.strip))
        ws = ws.filter(lambda w: w[0] in ts)
    return l2id, ws

In [ ]:
l2id, ws = await _get_flakes()

In [ ]:
ws

[('UndefinedName', '01_flakes', (3, 2), "undefined name 'FC'", ('FC',)), ('UndefinedName', '01_flakes', (4, 20), "undefined name 'Message'", ('Message',)), ('UndefinedName', '01_flakes', (6, 20), "undefined name 'Reporter'", ('Reporter',)), ('UndefinedName', '01_flakes', (8, 25), "undefined name 'FC'", ('FC',)), ('UndefinedName', '01_flakes', (9, 26), "undefined name 'StringIO'", ('StringIO',)), ('UndefinedName', '01_flakes', (9, 38), "undefined name 'StringIO'", ('StringIO',)), ('UndefinedName', '01_flakes', (19, 27), "undefined name 're'", ('re',)), ('UndefinedName', '01_flakes', (24, 21), "undefined name 'FC'", ('FC',)), ('UndefinedName', '01_flakes', (27, 60), "undefined name 'repeat'", ('repeat',)), ('UndefinedName', '01_flakes', (38, 22), "undefined name 'find_msgs'", ('find_msgs',)), ('UndefinedName', '01_flakes', (39, 45), "undefined name 'is_exported'", ('is_exported',)), ('UndefinedName', '01_flakes', (45, 13), "undefined name 'pyflakes'", ('pyflakes',)), ('UndefinedName', '0

In [ ]:
srclines = l2id.itemgot(0)
print(f"...\n{'\n'.join(srclines[10:30])}\n...")

...
    def close(self): self._stderr.close(); self._stdout.close()
    def unexpectedError(self, filename, msg): self.warnings.append(('UnexpectedError', filename, (0,0), msg, ()))
    def syntaxError(self, filename, msg, lineno, offset, text):
        line = None if text is None else text.splitlines()[-1]
        lineno = max(lineno or 0, 1)
        if offset is not None: offset = max(offset, 1)
        rng, s = (lineno, offset or 0), ''
        if line is not None:
           s = f"{line}\n{re.sub(r'\S', ' ', line[:offset-1])+"^\n" if offset is not None else ''}"
        self.warnings.append(('SyntaxError', filename, rng, msg, (s,)))
# %% msgid: _28266a58
def _line2msgid(codes, msgids):
    "Return list of (code_line, line_num_in_msg, msgid) tuples for each concatenated source line"
    line, l2id = 1, FC.L()
    for code, msgid in zip(codes, msgids):
        codelns = f"# %% msgid: {msgid}\n{code}\n".split('\n')
        l2id.extend(zip(codelns, range(0, len(codelns)-1), repeat(msgi

In [ ]:
#| export
async def check_flakes(
    dname:str='',  # dialog name (relative like 'data/test' or absolute), defaults to current
    wtypes:str=''  # filter: IMPORTS, VARS, or comma-separated warning types like 'UnusedImport,UndefinedName'; defaults to all types
) -> list:  # [{'wtype':'UnusedImport', 'description':"'os' imported but unused", 'id':'_abc123', 'line_n':5}, ...]
    "Check exported code for pyflakes issues"
    l2id, ws = await _get_flakes(dname, wtypes)
    return [dict(wtype=w[0], description=w[3], id=l2id[w[2][0]-1][2], line_n=l2id[w[2][0]-1][1]) for w in ws]

In [ ]:
wrns = await check_flakes(_DREL, wtypes='UnusedImport')
print(wrns)

[{'wtype': 'UnusedImport', 'description': "'fastcore.all as FC' imported but unused", 'id': '_e72f67fd', 'line_n': 5}]


In [ ]:
wrns = await check_flakes(_DABS)
for wrn in wrns: print(wrn)

{'wtype': 'UnusedImport', 'description': "'fastcore.all as FC' imported but unused", 'id': '_e72f67fd', 'line_n': 5}
{'wtype': 'ImportStarUsed', 'description': "'from dialoghelper import *' used; unable to detect undefined names", 'id': '_e72f67fd', 'line_n': 8}
{'wtype': 'ImportStarUsage', 'description': "'is_usable_tool' may be undefined, or defined from star imports: dialoghelper", 'id': '_9027bb28', 'line_n': 13}
{'wtype': 'ImportStarUsage', 'description': "'add_msg' may be undefined, or defined from star imports: dialoghelper", 'id': '_612c72ae', 'line_n': 4}
{'wtype': 'ImportStarUsage', 'description': "'mk_toollist' may be undefined, or defined from star imports: dialoghelper", 'id': '_612c72ae', 'line_n': 4}
{'wtype': 'ImportStarUsage', 'description': "'find_msg_id' may be undefined, or defined from star imports: dialoghelper", 'id': '_6a2bafc4', 'line_n': 5}
{'wtype': 'ImportStarUsage', 'description': "'add_msg' may be undefined, or defined from star imports: dialoghelper", 'id

In [ ]:
wrns = await check_flakes()
for wrn in wrns: print(wrn)


{'wtype': 'UndefinedName', 'description': "undefined name 'FC'", 'id': '_42c1d335', 'line_n': 2}
{'wtype': 'UndefinedName', 'description': "undefined name 'Message'", 'id': '_42c1d335', 'line_n': 3}
{'wtype': 'UndefinedName', 'description': "undefined name 'Reporter'", 'id': '_b6e1684c', 'line_n': 1}
{'wtype': 'UndefinedName', 'description': "undefined name 'FC'", 'id': '_b6e1684c', 'line_n': 3}
{'wtype': 'UndefinedName', 'description': "undefined name 'StringIO'", 'id': '_b6e1684c', 'line_n': 4}
{'wtype': 'UndefinedName', 'description': "undefined name 'StringIO'", 'id': '_b6e1684c', 'line_n': 4}
{'wtype': 'UndefinedName', 'description': "undefined name 're'", 'id': '_b6e1684c', 'line_n': 14}
{'wtype': 'UndefinedName', 'description': "undefined name 'FC'", 'id': '_28266a58', 'line_n': 3}
{'wtype': 'UndefinedName', 'description': "undefined name 'repeat'", 'id': '_28266a58', 'line_n': 6}
{'wtype': 'UndefinedName', 'description': "undefined name 'find_msgs'", 'id': '_0af65b34', 'line_n'

## pretty print

In [ ]:
#| export
def _group_flakes(ws, l2id):
    "Group pyflakes warnings by message id and warning type"
    res = defaultdict(lambda:defaultdict(list))
    for wtype, _, rng, *rest in ws:
        msgid = l2id[rng[0]-1][2]
        res[msgid][wtype].append((rng, rest))
    return res

In [ ]:
l2id, ws = await _get_flakes(_DREL)

grouped = _group_flakes(ws, l2id)
grouped

defaultdict(<function __main__._group_flakes.<locals>.<lambda>()>,
            {'_e72f67fd': defaultdict(list,
                         {'UnusedImport': [((6, 1),
                            ["'fastcore.all as FC' imported but unused",
                             ('fastcore.all as FC',)])],
                          'ImportStarUsed': [((9, 1),
                            ["'from dialoghelper import *' used; unable to detect undefined names",
                             ('dialoghelper',)])]}),
             '_9027bb28': defaultdict(list,
                         {'ImportStarUsage': [((62, 20),
                            ["'is_usable_tool' may be undefined, or defined from star imports: dialoghelper",
                             ('is_usable_tool', 'dialoghelper')])]}),
             '_612c72ae': defaultdict(list,
                         {'ImportStarUsage': [((68, 5),
                            ["'add_msg' may be undefined, or defined from star imports: dialoghelper",
                

In [ ]:
l2id, ws = await _get_flakes()

grouped = _group_flakes(ws, l2id)
grouped

defaultdict(<function __main__._group_flakes.<locals>.<lambda>()>,
            {'_42c1d335': defaultdict(list,
                         {'UndefinedName': [((3, 2),
                            ["undefined name 'FC'", ('FC',)]),
                           ((4, 20),
                            ["undefined name 'Message'", ('Message',)])]}),
             '_b6e1684c': defaultdict(list,
                         {'UndefinedName': [((6, 20),
                            ["undefined name 'Reporter'", ('Reporter',)]),
                           ((8, 25), ["undefined name 'FC'", ('FC',)]),
                           ((9, 26),
                            ["undefined name 'StringIO'", ('StringIO',)]),
                           ((9, 38),
                            ["undefined name 'StringIO'", ('StringIO',)]),
                           ((19, 27), ["undefined name 're'", ('re',)])]}),
             '_28266a58': defaultdict(list,
                         {'UndefinedName': [((24, 21),
                

In [ ]:
await read_msg(0, True, id='_42c1d335')

{'idle_evt': '<asyncio.locks.Event object at 0x700f5c29fdd0 [set]>',
 'id': '_42c1d335',
 'time_run': '2026-04-18T18:43:25.601948+00:00',
 'is_exported': 0,
 'skipped': 0,
 'bookmark': None,
 'i_collapsed': 0,
 'o_collapsed': 0,
 'heading_collapsed': False,
 'i_clamp': False,
 'o_clamp': False,
 'pinned': 0,
 'run': False,
 'oob': None,
 'scroll': None,
 'content': '#| exporti\n@FC.patch\ndef as_tuple(self: Message): return type(self).__name__, self.filename, (self.lineno, self.col+1), self.message % self.message_args, self.message_args',
 'output': '',
 'msg_type': 'code',
 'input_tokens': 96,
 'output_tokens': None}

In [ ]:
#| export
_lnks = (
    '''<span hx-on-click="setTimeout(() => selectMsg($('%s'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**%s**</span>  \n\n''',
    '''<h5 class="uk-flex"><a class="uk-link" href="%s"><strong>%s</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  \n\n'''
)

In [ ]:
s = ''
for msgid, wtypes in grouped.items():
    s += _lnks[1] % (msgid, msgid)#.format(msgid, msgid)
    for wtype, ws in wtypes.items():
        s += f"- **{wtype}**: {len(ws)}  \n"
        for w in ws: s += f"    `{w[0]}`: {w[1][0].replace("'", "`")}  \n"
    s += '\n'
print(s[:300])

<h5 class="uk-flex"><a class="uk-link" href="_42c1d335"><strong>_42c1d335</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  

- **UndefinedName**: 2  
    `(3, 2)`: undefined 


In [ ]:
dname = _DABS
rel = not dname or dname[0] != '/'
dname, rel, (str(Path(find_dname()).parent/dname) if rel else dname)[1:]

('/prj/pote/nbs/data/test', False, 'prj/pote/nbs/data/test')

In [ ]:
dname = _DREL
rel = not dname or dname[0] != '/'
dname, rel, (str(Path(find_dname()).parent/dname) if rel else dname)[1:]

('data/test', True, 'prj/pote/nbs/data/test')

In [ ]:
dname = find_dname()
rel = not dname or dname[0] != '/'
dname, rel, (str(Path(find_dname()).parent/dname) if rel else dname)[1:]

('/prj/pote/nbs/01_flakes', False, 'prj/pote/nbs/01_flakes')

In [ ]:
q = quote_plus((str(Path(find_dname()).parent/dname) if not dname or dname[0]!='/' else dname)[1:])
dlnk = f"/dialog_?name={q}"
dlnk

'/dialog_?name=prj%2Fpote%2Fnbs%2F01_flakes'

In [ ]:
l2id, ws = await _get_flakes()
grouped = _group_flakes(ws, l2id)
for msgid, wtypes in grouped.items():
    lnk = f"{dlnk}#{msgid}"
    s += _lnks[bool(dname)] % (lnk, msgid)
    for wtype, ws in wtypes.items():
        s += f"- **{wtype}**: {len(ws)}  \n"
        for w in ws: s += f"    `{w[0]}`: {w[1][0].replace("'", "`")}  \n"
    s += '\n'
s

'<h5 class="uk-flex"><a class="uk-link" href="_42c1d335"><strong>_42c1d335</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  \n\n- **UndefinedName**: 2  \n    `(3, 2)`: undefined name `FC`  \n    `(4, 20)`: undefined name `Message`  \n\n<h5 class="uk-flex"><a class="uk-link" href="_b6e1684c"><strong>_b6e1684c</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  \n\n- **UndefinedName**: 5  \n    `(6, 20)`: undefined name `Reporter`  \n    `(8, 25)`: undefined name `FC`  \n    `(9, 26)`: undefined name `StringIO`  \n    `(9, 38)`: undefined name `StringIO`  \n    `(19, 27)`: undefined name `re`  \n\n<h5 class="uk-flex"><a class="uk-link" href="_28266a58"><strong>_28266a58</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 

In [ ]:
#| export
def _render_flakes_report(dname, ws, l2id):
    dlnk, s = '', ''
    if dname:
        q = quote_plus((str(Path(find_dname()).parent/dname) if not dname or dname[0]!='/' else dname)[1:])
        dlnk = f"/dialog_?name={q}"
    grouped = _group_flakes(ws, l2id)
    for msgid, wtypes in grouped.items():
        lnk = f"{dlnk}#{msgid}"
        s += _lnks[bool(dname)] % (lnk, msgid)
        for wtype, ws in wtypes.items():
            s += f"- **{wtype}**: {len(ws)}  \n"
            for w in ws: s += f"    `{w[0]}`: {w[1][0].replace("'", "`")}  \n"
        s += '\n'
    return s

In [ ]:
l2id, ws = await _get_flakes(_DREL)
s = _render_flakes_report(_DREL, ws, l2id)
print(s[:400])

<h5 class="uk-flex"><a class="uk-link" href="/dialog_?name=prj%2Fpote%2Fnbs%2Fdata%2Ftest#_e72f67fd"><strong>_e72f67fd</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  

- **UnusedImport**: 1  
    `(6, 1)`: `fastcore.all as FC` imported but unused  
- **ImportStarUsed**: 1


In [ ]:
l2id, ws = await _get_flakes(_DABS)
s = _render_flakes_report(_DABS, ws, l2id)
print(s[:400])

<h5 class="uk-flex"><a class="uk-link" href="/dialog_?name=prj%2Fpote%2Fnbs%2Fdata%2Ftest#_e72f67fd"><strong>_e72f67fd</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  

- **UnusedImport**: 1  
    `(6, 1)`: `fastcore.all as FC` imported but unused  
- **ImportStarUsed**: 1


In [ ]:
l2id, ws = await _get_flakes()
s = _render_flakes_report('', ws, l2id)
print(s[:400])

<span hx-on-click="setTimeout(() => selectMsg($('#_42c1d335'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_42c1d335**</span>  

- **UndefinedName**: 2  
    `(3, 2)`: undefined name `FC`  
    `(4, 20)`: undefined name `Message`  

<span hx-on-click="setTimeout(() => selectMsg($('#_b6e1684c'), {centered: true}), 100)" class="uk-link text-blu


In [ ]:
cprint(s)

<span hx-on-click="setTimeout(() => selectMsg($('#_42c1d335'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_42c1d335**</span>  

- **UndefinedName**: 2  
    `(3, 2)`: undefined name `FC`  
    `(4, 20)`: undefined name `Message`  

<span hx-on-click="setTimeout(() => selectMsg($('#_b6e1684c'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_b6e1684c**</span>  

- **UndefinedName**: 5  
    `(6, 20)`: undefined name `Reporter`  
    `(8, 25)`: undefined name `FC`  
    `(9, 26)`: undefined name `StringIO`  
    `(9, 38)`: undefined name `StringIO`  
    `(19, 27)`: undefined name `re`  

<span hx-on-click="setTimeout(() => selectMsg($('#_28266a58'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_28266a58**</span>  

- **UndefinedName**: 2  
    `(24, 21)`: undefined name `FC`  
    `(27, 60)`: undefined name `repeat`  

<span hx-on-click="setTimeout(() => selectMsg($('#_0af65b34'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_0af65b34**</span>  

- **UndefinedName**: 2  
    `(38, 22)`: undefined name `find_msgs`  
    `(39, 45)`: undefined name `is_exported`  

<span hx-on-click="setTimeout(() => selectMsg($('#_5e564084'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_5e564084**</span>  

- **UndefinedName**: 1  
    `(45, 13)`: undefined name `pyflakes`  

<span hx-on-click="setTimeout(() => selectMsg($('#_84e3ec48'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_84e3ec48**</span>  

- **UndefinedName**: 1  
    `(62, 5)`: undefined name `_check`  

<span hx-on-click="setTimeout(() => selectMsg($('#_65ee988a'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_65ee988a**</span>  

- **UndefinedName**: 3  
    `(67, 35)`: undefined name `Path`  
    `(67, 49)`: undefined name `find_dname`  
    `(69, 18)`: undefined name `FC`  

<span hx-on-click="setTimeout(() => selectMsg($('#_b349e9cb'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_b349e9cb**</span>  

- **UndefinedName**: 2  
    `(83, 11)`: undefined name `defaultdict`  
    `(83, 30)`: undefined name `defaultdict`  

<span hx-on-click="setTimeout(() => selectMsg($('#_b8947210'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_b8947210**</span>  

- **UndefinedName**: 3  
    `(97, 13)`: undefined name `quote_plus`  
    `(97, 29)`: undefined name `Path`  
    `(97, 34)`: undefined name `find_dname`  

<span hx-on-click="setTimeout(() => selectMsg($('#_447c516f'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer 
hover:bg-muted truncate">**_447c516f**</span>  

- **UndefinedName**: 2  
    `(118, 11)`: undefined name `link_msg`  
    `(129, 13)`: undefined name `Markdown`

In [ ]:
Markdown(s)

<div class="prose">

<span hx-on-click="setTimeout(() => selectMsg($('#_42c1d335'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_42c1d335**</span>  

- **UndefinedName**: 2  
    `(3, 2)`: undefined name `FC`  
    `(4, 20)`: undefined name `Message`  

<span hx-on-click="setTimeout(() => selectMsg($('#_b6e1684c'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_b6e1684c**</span>  

- **UndefinedName**: 5  
    `(6, 20)`: undefined name `Reporter`  
    `(8, 25)`: undefined name `FC`  
    `(9, 26)`: undefined name `StringIO`  
    `(9, 38)`: undefined name `StringIO`  
    `(19, 27)`: undefined name `re`  

<span hx-on-click="setTimeout(() => selectMsg($('#_28266a58'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_28266a58**</span>  

- **UndefinedName**: 2  
    `(24, 21)`: undefined name `FC`  
    `(27, 60)`: undefined name `repeat`  

<span hx-on-click="setTimeout(() => selectMsg($('#_0af65b34'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_0af65b34**</span>  

- **UndefinedName**: 2  
    `(38, 22)`: undefined name `find_msgs`  
    `(39, 45)`: undefined name `is_exported`  

<span hx-on-click="setTimeout(() => selectMsg($('#_5e564084'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_5e564084**</span>  

- **UndefinedName**: 1  
    `(45, 13)`: undefined name `pyflakes`  

<span hx-on-click="setTimeout(() => selectMsg($('#_84e3ec48'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_84e3ec48**</span>  

- **UndefinedName**: 1  
    `(62, 5)`: undefined name `_check`  

<span hx-on-click="setTimeout(() => selectMsg($('#_65ee988a'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_65ee988a**</span>  

- **UndefinedName**: 3  
    `(67, 35)`: undefined name `Path`  
    `(67, 49)`: undefined name `find_dname`  
    `(69, 18)`: undefined name `FC`  

<span hx-on-click="setTimeout(() => selectMsg($('#_b349e9cb'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_b349e9cb**</span>  

- **UndefinedName**: 2  
    `(83, 11)`: undefined name `defaultdict`  
    `(83, 30)`: undefined name `defaultdict`  

<span hx-on-click="setTimeout(() => selectMsg($('#_b8947210'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_b8947210**</span>  

- **UndefinedName**: 3  
    `(97, 13)`: undefined name `quote_plus`  
    `(97, 29)`: undefined name `Path`  
    `(97, 34)`: undefined name `find_dname`  

<span hx-on-click="setTimeout(() => selectMsg($('#_447c516f'), {centered: true}), 100)" class="uk-link text-blue-600 p-1 cursor-pointer hover:bg-muted truncate">**_447c516f**</span>  

- **UndefinedName**: 2  
    `(118, 11)`: undefined name `link_msg`  
    `(129, 13)`: undefined name `Markdown`  



</div>

## all together now


In [ ]:
#| export
async def add_flakes(
    dname:str='',  # dialog name (relative like 'data/test' or absolute), defaults to current
    wtypes:str='' # filter: IMPORTS, VARS, or comma-separated warning types like 'UnusedImport,UndefinedName'; defaults to all types
):
    """Add a message with the warnings generated by pyflakes on the exported code.

Dialog names other than the default must be paths relative to the solveit root directory (if starting with `/`) or relative to the current dialog (if not starting with `/`), and should *not* include the .ipynb extension."""
    l2id, ws = await _get_flakes(dname, wtypes)
    s = _render_flakes_report(dname, ws, l2id)
    await link_msg(s)

async def show_flakes(
    dname:str='',  # dialog name (relative like 'data/test' or absolute), defaults to current
    wtypes:str='' # filter: IMPORTS, VARS, or comma-separated warning types like 'UnusedImport,UndefinedName'; defaults to all types
):
    """Show the warnings generated by pyflakes on the exported code.

Dialog names other than the default must be paths relative to the solveit root directory (if starting with `/`) or relative to the current dialog (if not starting with `/`), and should *not* include the .ipynb extension."""
    l2id, ws = await _get_flakes(dname, wtypes)
    s = _render_flakes_report(dname, ws, l2id)
    display(Markdown(s if s else 'No warnings to report'))

In [ ]:
await add_flakes(_DREL)

<h5 class="uk-flex"><a class="uk-link" href="/dialog_?name=prj%2Fpote%2Fnbs%2Fdata%2Ftest#_e72f67fd"><strong>_e72f67fd</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  

- **UnusedImport**: 1  
    `(6, 1)`: `fastcore.all as FC` imported but unused  
- **ImportStarUsed**: 1  
    `(9, 1)`: `from dialoghelper import *` used; unable to detect undefined names  

<h5 class="uk-flex"><a class="uk-link" href="/dialog_?name=prj%2Fpote%2Fnbs%2Fdata%2Ftest#_9027bb28"><strong>_9027bb28</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  

- **ImportStarUsage**: 1  
    `(62, 20)`: `is_usable_tool` may be undefined, or defined from star imports: dialoghelper  

<h5 class="uk-flex"><a class="uk-link" href="/dialog_?name=prj%2Fpote%2Fnbs%2Fdata%2Ftest#_612c72ae"><strong>_612c72ae</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  

- **ImportStarUsage**: 2  
    `(68, 5)`: `add_msg` may be undefined, or defined from star imports: dialoghelper  
    `(68, 13)`: `mk_toollist` may be undefined, or defined from star imports: dialoghelper  

<h5 class="uk-flex"><a class="uk-link" href="/dialog_?name=prj%2Fpote%2Fnbs%2Fdata%2Ftest#_6a2bafc4"><strong>_6a2bafc4</strong></a>&nbsp;<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" height="16px" width="16px" class="lucide-icon "><use href="#lc-external-link"></use></h5></svg>  

- **ImportStarUsage**: 2  
    `(74, 14)`: `find_msg_id` may be undefined, or defined from star imports: dialoghelper  
    `(76, 13)`: `add_msg` may be undefined, or defined from star imports: dialoghelper  


<!-- linkedto: _114c5311 -->

The first imports block at the beginning of the dialog is not exported. So `check_flakes` reports many `UndefinedName':

In [ ]:
await show_flakes()

<div class="prose">

No warnings to report

</div>

False negative. We're not detecting exported by directives.

Go now to the first import block and mark it for export. Come back here and run `add_flakes` below.

# Next

- Detect not only exported by `e`, also by directives (`#| exporti')

# export -

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# dlg_export()